In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from utils.dicece_loss import DiceCELoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.boundary_loss import BoundaryLoss
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch_cbam,
    validate_one_epoch_cbam,
    save_checkpoint,
    PatchDataset_cbam,
    evaluate_full_volume
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.65
NUM_WORKERS = 2
train_dataset = PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [6]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [7]:
class FinalLoss(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.dice_ce = DiceCELoss(weight=weight)
        self.boundary = BoundaryLoss()

    def forward(self, logits, targets):
        return self.dice_ce(logits, targets) + 0.3 * self.boundary(logits, targets)

In [8]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = FinalLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_21048\178764738.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")
    
    scheduler.step()

100%|██████████| 198/198 [18:58<00:00,  5.75s/it]



Epoch 0
Train Loss: 8.3763
Val Loss: 2.5872
Per Class Dice: [1.87597166e-01 1.11111669e-01 4.20555902e-08 5.60088571e-02
 2.16125199e-07 2.14469128e-01]
Saved Best Model


100%|██████████| 198/198 [18:17<00:00,  5.54s/it]



Epoch 1
Train Loss: 3.9439
Val Loss: 1.8935
Per Class Dice: [5.10050317e-01 1.11111115e-01 2.66594599e-08 2.69403399e-01
 6.33587885e-02 6.29361648e-01]
Saved Best Model


100%|██████████| 198/198 [17:05<00:00,  5.18s/it]


...


Epoch 36
Train Loss: 0.9356
Val Loss: 0.4564
Per Class Dice: [0.92211233 0.8330127  0.77987993 0.74587368 0.60432137 0.87390657]


100%|██████████| 198/198 [23:22<00:00,  7.09s/it]



Epoch 37
Train Loss: 1.0068
Val Loss: 0.4163
Per Class Dice: [0.93542839 0.87458505 0.84894692 0.74275869 0.79005634 0.85633952]


 13%|█▎        | 26/198 [03:41<24:28,  8.54s/it]


KeyboardInterrupt: 

In [10]:
#  RESUME TRAINING 

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

start_epoch = 0
best_val_loss = float("inf")

resume_path = os.path.join(SAVE_DIR, "best_model.pth")  

if os.path.exists(resume_path):
    print("Loading checkpoint:", resume_path)

    checkpoint = torch.load(resume_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    scheduler.last_epoch = start_epoch - 1

    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, starting fresh.")


Loading checkpoint: c:\Users\Yeseswini\OneDrive\Desktop\dhanush_mini\mini-project-1\experiments\cbam_nnunet_fold0\best_model.pth
Resuming from epoch 28


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_17656\3307613596.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_path, map_location=devic

In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    mean_dice = val_dice.mean()
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    scheduler.step()

100%|██████████| 198/198 [27:21<00:00,  8.29s/it]



Epoch 0
Train Loss: 1.0975
Val Loss: 0.4362
Per Class Dice: [0.80394886 0.86636375 0.83958332 0.66574638 0.80281573 0.88637714]
Mean Dice: 0.8108
Parotid Dice (L,R): 0.6657, 0.8028
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model


100%|██████████| 198/198 [24:48<00:00,  7.52s/it]



Epoch 1
Train Loss: 1.0475
Val Loss: 0.3108
Per Class Dice: [0.90832808 0.83128494 0.79248219 0.74198039 0.74091326 0.86230787]
Mean Dice: 0.8129
Parotid Dice (L,R): 0.7420, 0.7409
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model


100%|██████████| 198/198 [27:14<00:00,  8.25s/it]


...


Epoch 16
Train Loss: 0.9989
Val Loss: 0.3493
Per Class Dice: [0.72215295 0.85819423 0.89698588 0.91402337 0.90230053 0.89994286]
Mean Dice: 0.8656
Parotid Dice (L,R): 0.9140, 0.9023
Saved Best Parotid Model


100%|██████████| 198/198 [1:46:56<00:00, 32.41s/it]  



Epoch 17
Train Loss: 0.9347
Val Loss: 0.4448
Per Class Dice: [0.88556981 0.68958078 0.81731186 0.68618015 0.79656893 0.73759827]
Mean Dice: 0.7688
Parotid Dice (L,R): 0.6862, 0.7966


  8%|▊         | 15/198 [13:51<2:49:09, 55.46s/it]


KeyboardInterrupt: 

In [9]:
#  RESUME TRAINING 

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

start_epoch = 0
best_val_loss = float("inf")

resume_path = os.path.join(SAVE_DIR, "best_parotid_model.pth")  

if os.path.exists(resume_path):
    print("Loading checkpoint:", resume_path)

    checkpoint = torch.load(resume_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    scheduler.last_epoch = start_epoch - 1

    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, starting fresh.")


Loading checkpoint: c:\Users\Yeseswini\OneDrive\Desktop\dhanush_mini\mini-project-1\experiments\cbam_nnunet_fold0\best_parotid_model.pth


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_21048\2163492319.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_path, map_location=devic

Resuming from epoch 17


In [10]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    mean_dice = val_dice.mean()
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    scheduler.step()

100%|██████████| 198/198 [17:06<00:00,  5.18s/it]



Epoch 0
Train Loss: 0.9744
Val Loss: 0.3431
Per Class Dice: [0.91242616 0.80581462 0.79423939 0.8592731  0.8570413  0.82173479]
Mean Dice: 0.8418
Parotid Dice (L,R): 0.8593, 0.8570
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model


100%|██████████| 198/198 [17:28<00:00,  5.29s/it]



Epoch 1
Train Loss: 0.9831
Val Loss: 0.3906
Per Class Dice: [0.79831636 0.80986234 0.81777253 0.78716276 0.81901914 0.85371178]
Mean Dice: 0.8143
Parotid Dice (L,R): 0.7872, 0.8190


100%|██████████| 198/198 [17:51<00:00,  5.41s/it]


...


Epoch 46
Train Loss: 0.7743
Val Loss: 0.3371
Per Class Dice: [0.92365476 0.8717937  0.86205775 0.81280029 0.88407391 0.89772118]
Mean Dice: 0.8754
Parotid Dice (L,R): 0.8128, 0.8841


100%|██████████| 198/198 [16:52<00:00,  5.11s/it]



Epoch 47
Train Loss: 0.7715
Val Loss: 0.3721
Per Class Dice: [0.87558827 0.826246   0.79143533 0.88055438 0.82414976 0.84423009]
Mean Dice: 0.8404
Parotid Dice (L,R): 0.8806, 0.8241


 71%|███████   | 141/198 [12:18<04:58,  5.24s/it]


KeyboardInterrupt: 